In [ ]:
### AI gen code for converting the gz files to parquet

shape: (3, 2)
┌─────┬─────┐
│ a   ┆ b   │
│ --- ┆ --- │
│ i64 ┆ str │
╞═════╪═════╡
│ 1   ┆ x   │
│ 2   ┆ y   │
│ 3   ┆ z   │
└─────┴─────┘


In [6]:
from pathlib import Path
import subprocess
import polars as pl


def get_decompressor_cmd(path: Path) -> list[str]:
    """
    Use system decompression, not Polars decompression.
    """
    suffixes = "".join(path.suffixes)

    if suffixes.endswith(".csv.gz") or path.suffix == ".gz":
        return ["gzip", "-dc", str(path)]

    if suffixes.endswith(".csv.bz2") or path.suffix == ".bz2":
        return ["bzip2", "-dc", str(path)]

    raise ValueError(f"Unsupported compressed file type: {path}")


def convert_compressed_csv_to_parquet_parts(
    input_file: str | Path,
    output_dir: str | Path,
    *,
    rows_per_part: int = 1_000_000,
    temp_dir: str | Path | None = None,
    infer_schema_length: int = 0,
    parquet_compression: str = "zstd",
    separator: str = ",",
    has_header: bool = True,
    headers: list[str] | None = None,
):
    input_file = Path(input_file)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if rows_per_part <= 0:
        raise ValueError("rows_per_part must be positive")

    if headers is not None:
        if not headers:
            raise ValueError("headers cannot be an empty list")

        if len(headers) != len(set(headers)):
            raise ValueError("headers must be unique")

    if temp_dir is None:
        temp_dir = output_dir / "_tmp_csv_chunks"
    else:
        temp_dir = Path(temp_dir)

    temp_dir.mkdir(parents=True, exist_ok=True)

    cmd = get_decompressor_cmd(input_file)

    print(f"Input:  {input_file}")
    print(f"Output: {output_dir}")
    print(f"Temp:   {temp_dir}")
    print(f"Rows per part: {rows_per_part:,}")
    print(f"Parquet compression: {parquet_compression}")

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        bufsize=1024 * 1024,
    )

    if proc.stdout is None:
        raise RuntimeError("Could not open decompressor stdout")

    part = 0
    total_rows = 0

    try:
        source_header = b""

        # If the source file has a header, consume it once.
        # If custom headers are supplied, we discard this source header.
        if has_header:
            source_header = proc.stdout.readline()
            if not source_header:
                raise RuntimeError("Input file appears to be empty")

        while True:
            tmp_csv = temp_dir / f"part-{part:05d}.csv"
            out_parquet = output_dir / f"part-{part:05d}.parquet"

            rows_written = 0

            with tmp_csv.open("wb") as f:
                # Only write the source header if we are using the file's own headers.
                # If headers is supplied, Polars will assign those names directly.
                if has_header and headers is None:
                    f.write(source_header)

                for line in proc.stdout:
                    f.write(line)
                    rows_written += 1

                    if rows_written >= rows_per_part:
                        break

            if rows_written == 0:
                tmp_csv.unlink(missing_ok=True)
                break

            scan_kwargs = {
                "has_header": has_header and headers is None,
                "separator": separator,
                "infer_schema_length": infer_schema_length,
            }

            if headers is not None:
                scan_kwargs["new_columns"] = headers

            lf = pl.scan_csv(
                tmp_csv,
                **scan_kwargs,
            )

            lf.sink_parquet(
                out_parquet,
                compression=parquet_compression,
            )

            tmp_csv.unlink(missing_ok=True)

            total_rows += rows_written

            print(
                f"Wrote {out_parquet.name}: "
                f"{rows_written:,} rows "
                f"({total_rows:,} total)"
            )

            part += 1

            if rows_written < rows_per_part:
                break

    finally:
        if proc.stdout:
            proc.stdout.close()

        return_code = proc.wait()

        stderr = ""
        if proc.stderr:
            stderr = proc.stderr.read().decode("utf-8", errors="replace")

        if return_code != 0:
            raise RuntimeError(
                f"Decompressor failed with exit code {return_code}:\n{stderr}"
            )

    print("Conversion complete.")
    print(f"Parquet parts written to: {output_dir}")

colnames = ['time', 'source_user@domain', 'destination_user@domain', 'source_computer', 'destination_computer', 'authentication_type', 'logon_type', 'authentication_orientation', 'success/failure']

convert_compressed_csv_to_parquet_parts(
    input_file="auth.txt.gz",
    output_dir="parquet_output",
    rows_per_part=50_000_000,
    parquet_compression="zstd",
    infer_schema_length=0,
    separator=",",
    has_header=False,
    headers=colnames,
)

Input:  auth.txt.gz
Output: parquet_output
Temp:   parquet_output/_tmp_csv_chunks
Rows per part: 50,000,000
Parquet compression: zstd
Wrote part-00000.parquet: 50,000,000 rows (50,000,000 total)
Wrote part-00001.parquet: 50,000,000 rows (100,000,000 total)
Wrote part-00002.parquet: 50,000,000 rows (150,000,000 total)
Wrote part-00003.parquet: 50,000,000 rows (200,000,000 total)
Wrote part-00004.parquet: 50,000,000 rows (250,000,000 total)
Wrote part-00005.parquet: 50,000,000 rows (300,000,000 total)
Wrote part-00006.parquet: 50,000,000 rows (350,000,000 total)
Wrote part-00007.parquet: 50,000,000 rows (400,000,000 total)
Wrote part-00008.parquet: 50,000,000 rows (450,000,000 total)
Wrote part-00009.parquet: 50,000,000 rows (500,000,000 total)
Wrote part-00010.parquet: 50,000,000 rows (550,000,000 total)
Wrote part-00011.parquet: 50,000,000 rows (600,000,000 total)
Wrote part-00012.parquet: 50,000,000 rows (650,000,000 total)
Wrote part-00013.parquet: 50,000,000 rows (700,000,000 total)